In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/aieducation/what-course-are-you-going-to-take/src/small_bench/checkpoints/pre_cell_26.pickle

In [ ]:
%%cudf.pandas.profile
### cell 26 ###

def extract_features_from_dataframe(df, group_column, agg_function="mean"):
    """Extracts features for predicting the execution time of a groupby operation using GPU-friendly ops."""
    # number of rows and columns
    n_rows, n_cols = df.shape

    # number of groups and max group size
    n_groups = df[group_column].nunique()
    group_sizes = df[group_column].value_counts()
    max_group_size = group_sizes.max()

    # count columns by dtype without moving back to CPU
    # cast dtypes to string so value_counts works entirely on GPU
    dtype_counts = df.dtypes.astype(str).value_counts()
    int_count = dtype_counts.get('int64', 0) + dtype_counts.get('int32', 0)
    float_count = dtype_counts.get('float64', 0) + dtype_counts.get('float32', 0)
    str_count = dtype_counts.get('object', 0) + dtype_counts.get('string', 0)

    # encode aggregation function
    aggregation_mapping = {"sum": 0, "mean": 1, "count": 2}
    agg_function_encoded = aggregation_mapping.get(agg_function, 1)

    return {
        'n_rows': n_rows,
        'n_cols': n_cols,
        'group_range': n_groups,
        'n_groups': n_groups,
        'max_group_size': max_group_size,
        'int_count': int_count,
        'float_count': float_count,
        'str_count': str_count,
        'agg_function': agg_function_encoded
    }

# example call remains the same
extract_features_from_dataframe(course, "course_unit", "mean")